In [0]:
%python
!pip install pypdf
!pip install langchain-text-splitters
dbutils.library.restartPython()

In [0]:
%python
SOURCE_PATH = '/Volumes/ai2605/ai/unstructured/pdfs/'
import os
import json
import pandas as pd
import numpy as np
from pyspark.sql.types import *
from pyspark.sql.functions import *


raw_files_pdf = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .option("recursiveFileLookup", "true")
    .load(SOURCE_PATH)
)
display(raw_files_pdf)


In [0]:
%python
import pyspark.sql.functions as func
from typing import TypedDict, Dict
from functools import partial
from pyspark.sql.types import StringType, MapType, StructType, StructField
import io
import pypdf

class ParserReturnValue(TypedDict):
    doc_parsed_contents: Dict[str, str]
    product_category: str
    parser_status: str

def parse_byte_json(
    raw_doc_contents_bytes: bytes, document_path: str, content_key: str) -> ParserReturnValue:


    # Check if data is a PDF (starts with %PDF)
    if raw_doc_contents_bytes.startswith(b'%PDF'):
        try:
            pdf_file = io.BytesIO(raw_doc_contents_bytes)
            reader = pypdf.PdfReader(pdf_file)
            # Extract text from all pages
            text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
            
            return {
                "product_category": "PDF_DOCUMENT",
                "parser_status": "SUCCESS",
                "doc_parsed_contents": {"parsed_content": text}
            }
        except Exception as e:
            return {"product_category": "ERROR", "parser_status": f"PDF Error: {str(e)}", "doc_parsed_contents": {}}

    ''' 
    try:
        json_str = raw_doc_contents_bytes.decode("latin-1")
        json_data = json.loads(json_str)
        parsed_content = json_data[content_key]
        del json_data[content_key]
        json_data['parsed_content'] = parsed_content
        output = json_data
        status = 'SUCCESS'
    except json.JSONDecodeError as e:
        status = f"JSON decoding failed: {e}, {json_str[:50].strip()}"
        output = {"parsed_content": ""}
    except Exception as e:
        status = f"Unexpected errors: {e}, Input type: {type(raw_doc_contents_bytes)}"
        output = {"parsed_content": ""}
    return {
        "doc_parsed_contents": output,
        "product_category": "product_category",
        "parser_status": status
    }
'''

parser_udf = func.udf(
    partial(
        parse_byte_json,
        content_key='html_content'
    ), returnType=StructType([
        StructField("product_category", StringType(), True),
        StructField("parser_status", StringType(), True),
        StructField("doc_parsed_contents", MapType(StringType(), StringType()), True)
    ])
)

In [0]:
%python
parsed_files_staging_df = raw_files_pdf.withColumn(
    "parsed_contents",
    parser_udf(
        func.col("content"),
        func.col("path")))
#parsed_files_df = parsed_files_staging_df.filter(func.col("parser_status") == "SUCCESS")
#display(parsed_files_staging_df)
parsed_files_staging_df.write.mode("overwrite").saveAsTable("tbl_parsed_files")

In [0]:
%python
CHUNK_SIZE_TOKENS = 500
CHUNK_OVERLAP_TOKENS = 256
       
from langchain_text_splitters import RecursiveCharacterTextSplitter
class ChunkerReturnValue(TypedDict):
    chunks: list[str]
    chunker_status: str

def chunk_parsed_contents(  
    parsed_contents: str, chunk_size_tokens: int = CHUNK_SIZE_TOKENS, chunk_overlap_tokens: int = CHUNK_OVERLAP_TOKENS
) -> ChunkerReturnValue:
    
    if isinstance(parsed_contents, dict):
        text_to_chunk = parsed_contents.get('parsed_content', "")
    else:
        text_to_chunk = str(parsed_contents)
            
        # 2. Safety check: ensure it's actually a string now
    if not text_to_chunk or not isinstance(text_to_chunk, str):
        return {"chunks": [], "chunker_status": "Content is empty or not string"}

    """
    Split the parsed contents into chunks of a specified size and overlap.
    """
    try:
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size_tokens,
            chunk_overlap=chunk_overlap_tokens,
            separators=["\n\n", "\n", " ", ""],
            length_function=len,
        )
        chunks = text_splitter.split_text(parsed_contents)
        return {
            "chunks": [doc for doc in chunks],
            "chunker_status": "SUCCESS",
        }
    except Exception as e:
        return {
            "chunks": [],
            "chunker_status": f"Chunking failed: {e}",
        }
chunker_udf = func.udf(
    partial(chunk_parsed_contents, chunk_size_tokens=CHUNK_SIZE_TOKENS, chunk_overlap_tokens=CHUNK_OVERLAP_TOKENS),
    returnType=StructType([
        StructField("chunks", ArrayType(StringType()), True),
        StructField("chunker_status", StringType(), True),
    ])
)
text_to_chunk = func.col("parsed_contents.doc_parsed_contents.parsed_content")

chunked_files_df = parsed_files_staging_df.withColumn(
    "chunks",
    chunker_udf(func.col("parsed_contents.doc_parsed_contents.parsed_content")),
)

chunked_files_df = chunked_files_df.select("path", 
                                           func.explode("chunks.chunks").alias("chunked_text"), 
                                           func.md5("chunked_text").alias("chunk_id"))

display(chunked_files_df)

In [0]:
%python
(
chunked_files_df.write.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable("tbl_chunked_files")
)

spark.sql(f"alter table tbl_chunked_files set tblproperties (delta.enableChangeDataFeed = true)")
       
